# Error Case 4 — 403 Forbidden: No Catalog Role

**Error:** `403 Forbidden`  
**Root cause:** Principal has no catalog role for the requested catalog  
**Fix:** Assign catalog role to principal role

## Flow
```
GET /catalog/v1/{catalog}/namespaces
  → Auth: JWT valid ✅
  → RBAC: lookup grant_records for principal
  → 0 rows → no catalog role assigned
  → 403 ForbiddenException
```

In [6]:
import sys
sys.path.insert(0, '..')
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [7]:
# Cell 2: Create catalog + principal (NO catalog role assigned)
r1 = create_catalog()
assert r1.status_code == 201, r1.text

r2 = create_principal()
assert r2.status_code == 201
cred = r2.json()['credentials']

r3 = create_principal_role()
assert r3.status_code == 201

# Assign principal role to principal (but NOT catalog role)
r4 = assign_principal_role_to_principal()
assert r4.status_code == 201

print('Setup: catalog ✅ | principal ✅ | principal role ✅')
print('Missing: catalog role NOT assigned to principal role')

Setup: catalog ✅ | principal ✅ | principal role ✅
Missing: catalog role NOT assigned to principal role


In [8]:
# Cell 3: Get token for principal (no catalog role)
r_tok = get_token(cred['clientId'], cred['clientSecret'])
assert r_tok.status_code == 200
principal_token = r_tok.json()['access_token']

# Trigger 403 — no catalog role
r = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token)
)
print_response(r, '403 No catalog role response')
request_id = get_request_id(r)
assert r.status_code == 403
print(f'\nPolaris-Request-Id: {request_id}')
print('\n✅ 403 confirmed — no catalog role')


403 No catalog role response:
  Status:     403
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121
  Body: {
  "error": {
    "message": "Principal 'test-principal' with activated PrincipalRoles '[test-principal-role]' and activated grants via '[test-principal-role]' is not authorized for op LIST_NAMESPACES",
    "type": "ForbiddenException",
    "code": 403
  }
}

Polaris-Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121

✅ 403 confirmed — no catalog role


In [10]:
# Cell 4: Find log in OpenSearch
wait_for_logs(5)
hits = search_and_debug(request_id, minutes_ago=10)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
🔍 Searching for requestId: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121
✅ Found by requestId
Found 20 log entries:

ℹ️  [2026-06-02T00:58:14.819Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121
   realmId   : POLARIS
   message   : 192.168.194.1 - test-principal [02/Jun/2026:00:58:14 +0000] "GET /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 403 233

ℹ️  [2026-06-02T00:58:14.819Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121
   realmId   : POLARIS
   message   : 192.168.194.1 - test-principal [02/Jun/2026:00:58:14 +0000] "GET /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 403 233

ℹ️  [2026-06-02T00:58:14.818Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000121
   realm

In [11]:
# Cell 5: Fix — assign catalog role
r1 = create_catalog_role()
assert r1.status_code == 201

r2 = grant_privilege(privilege='CATALOG_MANAGE_CONTENT')
assert r2.status_code == 201

r3 = assign_catalog_role_to_principal_role()
assert r3.status_code == 201

print('Fix applied: catalog role created + assigned')

# Verify — get fresh token
r_tok = get_token(cred['clientId'], cred['clientSecret'])
principal_token = r_tok.json()['access_token']

r_verify = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token)
)
print(f'\nVerify: {r_verify.status_code}')
assert r_verify.status_code == 200
print('\n✅ Fixed — catalog accessible')

cleanup()
print('\n=== Summary ===')
print('Error:    403 Forbidden')
print('Cause:    No catalog role in grant_records')
print('Detect:   WARN message contains catalog role/Forbidden')
print('Fix:      Create catalog role → assign privilege → assign to principal role')

Fix applied: catalog role created + assigned

Verify: 200

✅ Fixed — catalog accessible
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    403 Forbidden
Cause:    No catalog role in grant_records
Detect:   WARN message contains catalog role/Forbidden
Fix:      Create catalog role → assign privilege → assign to principal role
